# Optimal price: grid search

In Radiant: **Load example > Optimal price (grid search)**.

## Question

> Demand for an ice cream bar is 1000 - 80 x price and each bar costs $2 to make. Which price between $4 and $10 maximizes profit? Use a grid search over price in steps of $0.01.

In [1]:
import pyrsm as rsm

A **grid** variable runs from a minimum to a maximum in steps. With grid variables the number of rows is the number of grid values (601 prices), not the number of simulations. `arg_max(profit)` is the row with the highest profit, so `price[arg_max(profit)]` is the best price; `max(profit)` is one number, the highest profit.

In [2]:
spec = {
    "name": "pricing",
    "runs": 1000,
    "seed": 1234,
    "variables": [
        {"name": "cost", "kind": "constant", "value": 2},
        {"name": "price", "kind": "grid", "min": 4, "max": 10, "step": 0.01},
    ],
    "formulas": [
        "demand = 1000 - 80 * price",
        "profit = demand * (price - cost)",
        "# the price in the row with the highest profit",
        "optimal_price = price[arg_max(profit)]",
        "max_profit = max(profit)",
    ],
}

In [3]:
sim = rsm.decide.simulate(spec)
sim.summary()

Simulation
Simulations: 601
Random seed: 1234
Sim data   : pricing
Constant   : cost 2
Grid search: price 4 10 0.01
Formulas   :
	demand = 1000 - 80 * price
	profit = demand * (price - cost)
	# the price in the row with the highest profit
	optimal_price = price[arg_max(profit)]
	max_profit = max(profit)

Constants:
   cost optimal_price max_profit
 2.0000        7.2500 2,205.0000

Variables:
        n_obs       mean       sd        min        p25     median        p75        max
 price    601     7.0000   1.7364     4.0000     5.5000     7.0000     8.5000    10.0000
 demand   601   440.0000 138.9104   200.0000   320.0000   440.0000   560.0000   680.0000
 profit   601 1,959.2000 226.4700 1,360.0000 1,800.0000 2,025.0000 2,160.0000 2,205.0000


In [4]:
import polars as pl

sim.data.sort("profit", descending=True).head(3)

shape: (3, 6)
┌───────┬──────┬────────┬──────────┬───────────────┬────────────┐
│ price ┆ cost ┆ demand ┆ profit   ┆ optimal_price ┆ max_profit │
│ ---   ┆ ---  ┆ ---    ┆ ---      ┆ ---           ┆ ---        │
│ f64   ┆ f64  ┆ f64    ┆ f64      ┆ f64           ┆ f64        │
╞═══════╪══════╪════════╪══════════╪═══════════════╪════════════╡
│ 7.25  ┆ 2.0  ┆ 420.0  ┆ 2205.0   ┆ 7.25          ┆ 2205.0     │
│ 7.26  ┆ 2.0  ┆ 419.2  ┆ 2204.992 ┆ 7.25          ┆ 2205.0     │
│ 7.24  ┆ 2.0  ┆ 420.8  ┆ 2204.992 ┆ 7.25          ┆ 2205.0     │
└───────┴──────┴────────┴──────────┴───────────────┴────────────┘

Two grid variables give every combination: with `x` from 1 to 3 and `y` from 4 to 5, the first varies fastest.

In [5]:
rsm.decide.simulate({"variables": [
    {"name": "x", "kind": "grid", "min": 1, "max": 3, "step": 1},
    {"name": "y", "kind": "grid", "min": 4, "max": 5, "step": 1},
]}).data

shape: (6, 2)
┌─────┬─────┐
│ x   ┆ y   │
│ --- ┆ --- │
│ f64 ┆ f64 │
╞═════╪═════╡
│ 1.0 ┆ 4.0 │
│ 2.0 ┆ 4.0 │
│ 3.0 ┆ 4.0 │
│ 1.0 ┆ 5.0 │
│ 2.0 ┆ 5.0 │
│ 3.0 ┆ 5.0 │
└─────┴─────┘